### Phase V Work Report: Liquidity State Reconstruction and Price Impact Dynamics

MY goal in this phase was to reconstruct the granular, pre-trade liquidity state of the Uniswap V3 pools and compute the exact price impact of every executed swap. To achieve this, I needed to extract foundational liquidity events (mints, burns, and initializations) and mathematically align them with the historical swap panel, enabling a precise analysis of slippage and market depth.

### Methodology

1. **Event Extraction and Cryptographic Decoding:** I engineered a custom scanner utilizing a direct Ethereum Remote Procedure Call to extract `Initialize`, `Mint`, and `Burn` events for all validated pools. Because Infura enforces strict payload limits on log queries, I implemented a recursive bisection algorithm. If a block range query failed due to excessive size, the scanner dynamically divided the block range in half and retried, ensuring zero data loss during high-volatility periods. Furthermore, I authored custom hexadecimal decoders to parse the 32-byte EVM data structures into standard integers, rigorously handling 256-bit signed overlaps.

2. **Liquidity Positioning and Tick Alignment:** To contextualize the liquidity events, I executed a backward-looking temporal join against the master swap ledger. By matching mint and burn events to the most recent preceding swap or initialization, I derived the exact active tick at the moment the liquidity was modified. This allowed me to definitively flag whether a liquidity position was deployed “in range” or “out of range” relative to the active market price.

3. **Pre-Trade State Reconstruction:** To calculate precise price impacts, it was imperative to know the exact state of the pool immediately before a transaction executed. I concatenated the initialization and swap events into a unified temporal ledger and applied a shifted window function partitioned by pool address. This lag operation successfully appended the pre-trade square root price and liquidity depth to every individual swap.

4. **Price Impact Mathematical Formulation:** Utilizing the canonical Uniswap V3 pricing formulas, I converted the pre-trade and post-trade square root values into standard decimal mid prices. I then calculated the effective execution price directly from the ratio of the transferred token amounts. The price impact was subsequently defined as the absolute percentage deviation of the effective execution price from the pre-trade mid price.


In [1]:
from config import OUT
import polars as pl
from pathlib import Path
from web3 import Web3

print(f"Saving data to: {OUT}")

Saving data to: C:\Users\Pouyan\python\thesis\Proposal\FINAL\Thesis_Output


---
## Initialization and Cryptographic Primitives

This cell establishes the foundational components required to parse Uniswap V3 liquidity events. It isolates the Keccak-256 cryptographic signatures for pool initializations, liquidity mints, and liquidity burns. It also defines the fundamental bitwise operations necessary to convert Ethereum hexadecimal payloads into signed integer formats for subsequent tick limit evaluations.

In [4]:
# load the validated pool registry from the previous notebook
D = pl.read_parquet("./Thesis_Output/pools_v1.parquet")

# define the cryptographic signatures for liquidity events
TOPICS = {
  "0x98636036cb66a9c19a37435efc1e90142190214e8abeb821bdba3f2990dd4c95": "initialize",
  "0x7a53080ba414158be7ec69b987b5fb7d07dee101fe85488f0853ae16239d0bde": "mint",
  "0x0c396cd989a39f4459b5fa1aed6a9a8dcdbc45908acfd67e028cd568da98982c": "burn",
}

# isolate only the pools that passed our safety and activity filters
POOLS = set(D.filter(pl.col("traded_in_window") & pl.col("price_safe"))["pool_address"])

def s24(word_hex):
    # extracts a signed 24 bit integer from a 32 byte hexadecimal word
    v = int(word_hex, 16) & 0xFFFFFF
    return v - (1 << 24) if v & (1 << 23) else v

def words(data_hex):
    # segments the raw data string into uniform 64 character blocks
    h = data_hex[2:]
    return [h[i:i+64] for i in range(0, len(h), 64)]

---

## High Frequency Event Scanner and Hexadecimal Parsing

This cell constructs a highly resilient blockchain scanner. Because querying large block ranges often exceeds node provider payload limits, this scanner employs a recursive bisection algorithm. If a query fails, it splits the request in half and tries again, guaranteeing absolute data capture. Additionally, it houses the custom decoding logic to unpack the complex hexadecimal event data into human readable dictionaries, strictly validating the tick boundaries to prevent corrupted data from entering the pipeline

In [7]:
w3 = Web3(Web3.HTTPProvider("Infura RPC")) #replace with your own endpoint 

# network constants and structural limits
FACTORY_BLOCK      = 12_369_621
WIN_LO, WIN_HI     = 21_500_000, 25_431_199
I128_MIN, I128_MAX = -(1 << 127), (1 << 127) - 1
TICK_LIMIT         = 887_272
SPACING            = {100: 1, 500: 10, 3000: 60, 10000: 200}
BIG                = ("amount", "amount0", "amount1", "liquidity_delta", "sqrt_price_x96")

OUT = Path("Dataset_E/raw")
OUT.mkdir(parents=True, exist_ok=True)

BY_LABEL = {v: k for k, v in TOPICS.items()}
INIT_T, MINT_T, BURN_T = BY_LABEL["initialize"], BY_LABEL["mint"], BY_LABEL["burn"]


# bitwise extraction primitives
def h0x(x):
    s = x.hex() if hasattr(x, "hex") else str(x)
    return s if s.startswith("0x") else "0x" + s

def u(w):     
    return int(w, 16)
def addr(w):  
    return "0x" + w[-40:].lower()
def fits(v):  
    return I128_MIN <= v <= I128_MAX
def hx(v):    
    return ("-0x" + format(-v, "x")) if v < 0 else "0x" + format(v, "x")


# explicit payload decoders
def dec_initialize(t, data):
    w = words(data)
    if len(t) != 1 or len(w) != 2:            
        return None, "shape"
    sp, tk = u(w[0]), s24(w[1])
    if sp == 0:                               
        return None, "sqrt_zero"
    if not (-TICK_LIMIT <= tk <= TICK_LIMIT): 
        return None, "tick_range"
    return {"sqrt_price_x96": sp, "tick": tk}, None

def dec_mint(t, data):
    w = words(data)
    if len(t) != 4 or len(w) != 4:            
        return None, "shape"
    lo, hi, amt = s24(t[2]), s24(t[3]), u(w[1])
    if lo >= hi:                              
        return None, "tick_order"
    if lo < -TICK_LIMIT or hi > TICK_LIMIT:   
        return None, "tick_range"
    if amt == 0:                              
        return None, "zero_mint"
    return {"kind": "mint", "owner": addr(t[1]), "sender": addr(w[0]),
            "tick_lower": lo, "tick_upper": hi,
            "amount": amt, "liquidity_delta": amt,
            "amount0": u(w[2]), "amount1": u(w[3])}, None

def dec_burn(t, data):
    w = words(data)
    if len(t) != 4 or len(w) != 3:            
        return None, "shape"
    lo, hi, amt = s24(t[2]), s24(t[3]), u(w[0])
    if lo >= hi:                              
        return None, "tick_order"
    if lo < -TICK_LIMIT or hi > TICK_LIMIT:   
        return None, "tick_range"
    return {"kind": "burn", "owner": addr(t[1]), "sender": None,
            "tick_lower": lo, "tick_upper": hi,
            "amount": amt, "liquidity_delta": -amt,
            "amount0": u(w[1]), "amount1": u(w[2])}, None

DECODERS = {"initialize": dec_initialize, "mint": dec_mint, "burn": dec_burn}


# disk writing mechanics
def _expand(r):
    o = {}
    for k, v in r.items():
        if k in BIG and v is not None:
            o["d_" + k]   = hx(v)
            o[k + "_f64"] = float(v)
            o[k + "_ovf"] = not fits(v)
            o[k]          = v if fits(v) else None
        else:
            o[k] = v
    return o

def _flush(rows, label, shard):
    ex = [_expand(r) for r in rows]
    ov = {c: pl.Int128 for c in BIG if c in ex[0]}
    df = pl.DataFrame(ex, schema_overrides=ov)
    df.write_parquet(OUT / f"{label}_{shard:04d}.parquet", compression="zstd")
    print(f"    wrote {label}_{shard:04d} rows {df.height:,}")


# resilient blockchain scanner
def scan(topic0, label, lo, hi, step=2_000, shard_rows=1_000_000, log_every=250_000):
    dec = DECODERS[label.split("_")[0]]
    rows, rej, shard, seen, kept = [], {}, 0, 0, 0
    t0, cur, nxt = time.time(), lo, lo + log_every
    
    while cur <= hi:
        top   = min(cur + step - 1, hi)
        stack = [(cur, top)]
        while stack:
            a, z = stack.pop()
            for attempt in range(3):
                try:
                    logs = w3.eth.get_logs({"fromBlock": a, "toBlock": z,
                                            "topics": [topic0]})
                    break
                except Exception as e:
                    # too many results trigger bisection split to avoid payload limits
                    if z > a:                       
                        m = (a + z) // 2
                        stack.append((m + 1, z))
                        stack.append((a, m))
                        logs = None
                        break
                    if attempt == 2: 
                        raise
                    time.sleep(1.5 * (attempt + 1))
            
            if logs is None:
                continue
                
            for L in logs:
                seen += 1
                t = [h0x(x) for x in L["topics"]]
                try:
                    r, why = dec(t, h0x(L["data"]))
                except Exception:
                    r, why = None, "malformed"
                if r is None:
                    rej[why] = rej.get(why, 0) + 1
                    continue
                
                pool = L["address"].lower()
                r |= {"pool_address": pool, "block_number": L["blockNumber"],
                      "log_index": L["logIndex"], "tx_hash": h0x(L["transactionHash"]),
                      "in_pools": pool in POOLS}
                rows.append(r)
                kept += 1
                
        cur = top + 1
        if len(rows) >= shard_rows:
            _flush(rows, label, shard)
            shard += 1
            rows = []
        if cur >= nxt:
            print(f"  {label} processing {cur:,} through {hi:,} seen {seen:,} kept {kept:,} duration {time.time()-t0:,.0f} seconds")
            nxt = cur + log_every
            
    if rows: 
        _flush(rows, label, shard)
    print(f"{label} completion seen {seen:,} kept {kept:,} rejected {rej}")
    return rej

print("system ready verifying primitives")

system ready verifying primitives


---

## Diagnostic Execution and Event Extraction

Before committing to a multi hour download process, this cell executes a rapid smoke test across a brief block window. It verifies that the parsed tick structures strictly align with the pool fee tiers spacing rules. Once the alignment is confirmed perfectly zero misaligned ticks, it deploys the full scale asynchronous scans to extract every relevant initialization, mint, and burn event.

In [10]:
# rapid diagnostic smoke test
for p in OUT.glob("mint_smoke_*"): 
    p.unlink()

scan(MINT_T, "mint_smoke", 21_500_000, 21_500_400, step=200)
sm = pl.read_parquet(OUT / "mint_smoke_*.parquet")

print(sm.select("pool_address","block_number","owner","tick_lower","tick_upper",
                "d_amount","d_amount0","d_amount1").head(3))

# verify that extracted ticks obey strict spacing modulo rules
chk = sm.join(D.select("pool_address","fee_tier"), on="pool_address") \
        .with_columns(pl.col("fee_tier").replace_strict(SPACING).alias("sp"))

print("misaligned ticks detected", chk.filter((pl.col("tick_lower") % pl.col("sp") != 0) |
                                (pl.col("tick_upper") % pl.col("sp") != 0)).height)

for p in OUT.glob("mint_smoke_*"): 
    p.unlink()

# full scale historical extraction
rej_init = scan(INIT_T, "initialize", FACTORY_BLOCK, WIN_HI, step=50_000)
rej_mint = scan(MINT_T, "mint",       WIN_LO,        WIN_HI, step=2_000)
rej_burn = scan(BURN_T, "burn",       WIN_LO,        WIN_HI, step=2_000)

    wrote mint_smoke_0000 rows 233
mint_smoke completion seen 233 kept 233 rejected {}
shape: (3, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ pool_addre ┆ block_numb ┆ owner      ┆ tick_lowe ┆ tick_uppe ┆ d_amount  ┆ d_amount0 ┆ d_amount1 │
│ ss         ┆ er         ┆ ---        ┆ r         ┆ r         ┆ ---       ┆ ---       ┆ ---       │
│ ---        ┆ ---        ┆ str        ┆ ---       ┆ ---       ┆ str       ┆ str       ┆ str       │
│ str        ┆ i64        ┆            ┆ i64       ┆ i64       ┆           ┆           ┆           │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 0x6dcba365 ┆ 21500001   ┆ 0xc36442b4 ┆ -887270   ┆ 887270    ┆ 0x6a8e055 ┆ 0x6a93dc7 ┆ 0x6a882e7 │
│ 7ee750a51a ┆            ┆ a4522e8713 ┆           ┆           ┆ 0ec5c6f   ┆ 47176ce   ┆ f560639   │
│ 13a235b4ed ┆            ┆ 99cd717abd ┆           ┆           ┆           ┆           ┆   

---
## Liquidity Positioning and Tick Alignment

This cell determines the exact state of the market when liquidity providers act. By performing a temporal backward join linking mint and burn events to the most recent known swap or initialization, the pipeline reconstructs the active market tick at the exact moment of the liquidity event. This mathematical mapping allows us to accurately classify if a provider was depositing capital directly into the active trading range or deploying out of bounds.

In [13]:
# file loading and temporal configuration
OUT = Path("Dataset_E/raw")
I = pl.read_parquet(OUT / "initialize_*.parquet").filter("in_pools")
M = pl.read_parquet(OUT / "mint_*.parquet").filter("in_pools")
B = pl.read_parquet(OUT / "burn_*.parquet").filter("in_pools")

# load the cleaned swap data using relative paths for portability
SWAPS = "./clean/*.parquet"

# create a unified temporal ordering integer based on block and log position
ORD = (pl.col("block_number").cast(pl.Int64) * 1_000_000
       + pl.col("log_index").cast(pl.Int64))

pool_ids = (pl.concat([
        pl.scan_parquet(SWAPS).select("pool_address"),
        M.lazy().select("pool_address"), B.lazy().select("pool_address"),
        I.lazy().select("pool_address")], how="vertical")
    .unique().collect().with_row_index("pid")
    .with_columns(pl.col("pid").cast(pl.UInt32)))

print(f"pool identification space {pool_ids.height:,}")

# market state consolidation
sw = (pl.scan_parquet(SWAPS).select("pool_address","block_number","log_index","tick")
        .join(pool_ids.lazy(), on="pool_address", how="inner")
        .select("pid", ORD.alias("ord"),
                pl.col("tick").cast(pl.Int32).alias("tick_pre"),
                pl.lit(False).alias("from_init")))

ini = (I.lazy().select("pool_address","block_number","log_index","tick")
        .join(pool_ids.lazy(), on="pool_address", how="inner")
        .select("pid", ORD.alias("ord"),
                pl.col("tick").cast(pl.Int32).alias("tick_pre"),
                pl.lit(True).alias("from_init")))

price = (pl.concat([sw, ini]).collect()
         .with_columns(pl.col("ord").alias("ord_src")).sort("ord"))
print(f"total price events {price.height:,}")

# backward temporal assignment
def attach(df):
    # merges liquidity events with the most recent preceding price state
    ovf = [c for c in ("amount0_ovf","amount1_ovf") if c in df.columns]
    flag = pl.any_horizontal([pl.col(c) for c in ovf]) if ovf else pl.lit(False)
    
    return (df.join(pool_ids, on="pool_address", how="left")
        .with_columns(ORD.alias("ord")).sort("ord")
        .join_asof(price, on="ord", by="pid", strategy="backward")
        .with_columns([
            pl.col("tick_pre").is_between(pl.col("tick_lower"),
                                          pl.col("tick_upper") - 1).alias("in_range"),
            (pl.col("amount") == 0).alias("is_poke"),
            ((pl.col("ord") - pl.col("ord_src")) // 1_000_000).alias("price_age_blocks"),
            flag.alias("amt_ovf")]))

ME, BE = attach(M), attach(B)

# generate diagnostic reporting to ensure high fidelity matching
for nm, df in (("mint", ME), ("burn", BE)):
    ok = df.filter(pl.col("tick_pre").is_not_null())
    print(f"{nm} metrics processed {df.height:,} matched {ok.height:,}")

print(f"mint in range percentage {ME.filter(pl.col('tick_pre').is_not_null())['in_range'].mean():.1%}")
print(f"burn in range percentage {BE.filter(pl.col('tick_pre').is_not_null() & ~pl.col('is_poke'))['in_range'].mean():.1%}")

ME.write_parquet(OUT.parent / "mint_priced.parquet", compression="zstd")
BE.write_parquet(OUT.parent / "burn_priced.parquet", compression="zstd")

pool identification space 41,761
total price events 64,027,081


C:\Users\Pouyan\AppData\Local\Temp\ipykernel_14512\3592787290.py:48: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  .join_asof(price, on="ord", by="pid", strategy="backward")


mint metrics processed 1,053,115 matched 1,053,115
burn metrics processed 1,260,050 matched 1,260,050
mint in range percentage 79.2%
burn in range percentage 60.8%


---
## Pre Trade State and Price Impact Formulation

This cell calculates the definitive econometric variables for the dataset. By stacking all initialization and swap events into a single timeline and using a lagging window function, it isolates the exact mathematical state of the pool immediately preceding every trade. It then executes the canonical Uniswap V3 pricing algorithms to derive the theoretical pre trade mid price, the actual execution price, and the precise mathematical price impact experienced by the trader.

In [16]:
SWAPS = "./clean/*.parquet"
OUT_E = Path("Dataset_E/Dataset_E_PoolState.parquet")

print("constructing pre trade states and computing price impacts")

# data ingestion and normalization
I = pl.read_parquet("Dataset_E/raw/initialize_*.parquet").select(
    "pool_address", 
    pl.col("block_number").cast(pl.Int64), 
    pl.col("log_index").cast(pl.Int64),
    pl.col("sqrt_price_x96_f64").alias("sqrt_price_x96"),
    pl.col("tick").cast(pl.Int64),
    pl.lit(0.0).alias("amount0_f64"),
    pl.lit(0.0).alias("amount1_f64"),
    pl.lit(True).alias("is_init")
)

S = pl.scan_parquet(SWAPS).select(
    "pool_address", 
    pl.col("block_number").cast(pl.Int64), 
    pl.col("log_index").cast(pl.Int64), 
    "transaction_hash",
    "sqrt_price_x96", 
    pl.col("tick").cast(pl.Int64), 
    "amount0_f64", "amount1_f64",
    "tx_from", "sender", "recipient", "liquidity_f64"
).with_columns(pl.lit(False).alias("is_init")).collect()

# pre trade state reconstruction
E = (pl.concat([I, S], how="diagonal")
     .sort(["pool_address", "block_number", "log_index"])
     .with_columns([
         # shift values backwards to attach the preceding state to the current swap
         pl.col("sqrt_price_x96").shift(1).over("pool_address").alias("pre_sqrt_price_x96"),
         pl.col("tick").shift(1).over("pool_address").alias("pre_tick"),
         pl.col("liquidity_f64").shift(1).over("pool_address").alias("pre_liquidity"),
     ])
     .filter(~pl.col("is_init")) 
     .drop("is_init")
)

# theoretical and execution pricing math
TWO_96 = 2.0 ** 96

# remove explicitly zero token transfers to prevent mathematical division failures
E = E.filter(pl.col("amount0_f64") != 0.0)

E = E.with_columns([
    # theoretical pool mid price prior to execution
    ((pl.col("pre_sqrt_price_x96") / TWO_96) ** 2).alias("price_pre"),
    
    # theoretical pool mid price immediately following execution
    ((pl.col("sqrt_price_x96") / TWO_96) ** 2).alias("price_post"),
    
    # actual effective price realized by the swapper
    (pl.col("amount1_f64").abs() / pl.col("amount0_f64").abs()).alias("price_exec")
])

# calculate the percentage deviation constituting the explicit price impact
E = E.with_columns(
    ((pl.col("price_exec") / pl.col("price_pre")) - 1).abs().alias("price_impact")
)

E = E.drop_nulls(subset=["price_pre", "price_exec"])
E.write_parquet(OUT_E, compression="zstd")

print(f"dataset e construction successful total rows {E.height:,}")

constructing pre trade states and computing price impacts
dataset e construction successful total rows 63,984,412


---
## Outlier Sanitization and Final Dataset Assembly
This concluding cell performs a critical econometric sanitation pass. Anomalous trades returning a price impact greater than one hundred percent usually represent severe Maximum Extractable Value sandwich attacks, dust division errors, or spam. Because these extreme outliers significantly skew regression coefficients, they are filtered out to preserve the integrity of standard market dynamics. The clean dataset is then exported for downstream statistical modeling.


In [19]:
0d


OUT_DIR = Path("./Thesis_Output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# econometric sanitation filter
# filter out extreme slippage events representing dust rounding or severe spam
E_clean = E.filter(pl.col("price_impact") <= 1.0)

removed = E.height - E_clean.height
print(f"removed {removed:,} extreme outlier rows representing {(removed/E.height):.4%} of the data")

# final dataset export
parquet_path = OUT_DIR / "Dataset_E_PoolState.parquet"
E_clean.write_parquet(parquet_path, compression="zstd")
print("saved optimized parquet format successfully")

csv_path = OUT_DIR / "Dataset_E_PoolState.csv"
print("writing csv format this will consume significant time and disk space")
E_clean.write_csv(csv_path)
print("saved universal csv format successfully")

print("clean price impact statistical overview")
print(E_clean.select("price_impact").describe())

removed 27,220 extreme outlier rows representing 0.0425% of the data
saved optimized parquet format successfully
writing csv format this will consume significant time and disk space
saved universal csv format successfully
clean price impact statistical overview
shape: (9, 2)
┌────────────┬──────────────┐
│ statistic  ┆ price_impact │
│ ---        ┆ ---          │
│ str        ┆ f64          │
╞════════════╪══════════════╡
│ count      ┆ 6.3957192e7  │
│ null_count ┆ 0.0          │
│ mean       ┆ 0.006726     │
│ std        ┆ 0.026869     │
│ min        ┆ 4.5648e-8    │
│ 25%        ┆ 0.000309     │
│ 50%        ┆ 0.00233      │
│ 75%        ┆ 0.005719     │
│ max        ┆ 1.0          │
└────────────┴──────────────┘


---
### Results & Data Integrity

The pipeline successfully isolated the liquidity dynamics and calculated price impacts for the entire transaction universe. I applied a necessary econometric filter to quarantine anomalous rows exhibiting price impacts greater than one hundred percent, which overwhelmingly represent dust division errors, spam, or extreme Maximum Extractable Value sandwich attacks that would otherwise distort the regression models. The final dataset was exported as **Dataset E**